# LeetCode #1110 — Delete Nodes And Return Forest

**Medium · Python 3 · Binary Tree, DFS, Hash Set**  
[Đề gốc LeetCode](https://leetcode.com/problems/delete-nodes-and-return-forest/) · [Tham khảo mô tả & giới hạn](https://leetcode.doocs.org/en/lc/1110/)

```python
class Solution:
    def delNodes(self, root: Optional[TreeNode], to_delete: List[int]) -> List[TreeNode]:
        ...
```

**Tinh thần học:** tự tìm ý tưởng xóa nút và cắt cạnh, xem trạng thái cây thay đổi, rồi mới đọc code tối ưu. Notebook có code thực thi, trace, các test và bài tập tự làm.

## 1. Mục tiêu

1. Hiểu `TreeNode`, `root`, cạnh cha–con và **forest (rừng)** là gì.
2. Phân biệt **xóa nút** với **xóa toàn bộ cây con**.
3. Biết dùng `set` để đánh dấu giá trị cần xóa và dùng DFS để duyệt cây.
4. Tự viết lời giải đệ quy và phiên bản dùng **stack** không phụ thuộc giới hạn đệ quy của Python.
5. Chứng minh tính đúng, tính độ phức tạp, kiểm tra các ca biên và tự làm bài tập.

## 2. Đề bài, ví dụ và ràng buộc

Cho một cây nhị phân `root` với các giá trị **đôi một khác nhau**, và danh sách `to_delete`. Xóa các nút có giá trị nằm trong danh sách đó. Những phần cây còn liên thông tạo thành một **rừng**; hãy trả về các **nút gốc** của những cây còn lại, **theo thứ tự bất kỳ**.

**Ví dụ 1:** `root=[1,2,3,4,5,6,7]`, `to_delete=[3,5]` → một đáp án là `[[1,2,None,4],[6],[7]]`.

```text
        1                    1
      /   \                /    (mất 3)
     2     3      -->      2          6      7
    / \   / \             /        (gốc)  (gốc)
   4   5 6   7            4
```

**Ví dụ 2:** `root=[1,2,4,None,3]`, `to_delete=[3]` → `[[1,2,4]]`.

Ràng buộc từ đề: tối đa **1000 nút**, giá trị nút khác nhau thuộc `[1,1000]`; danh sách xóa có tối đa 1000 giá trị khác nhau thuộc `[1,1000]`.

## 3. Nền tảng: nút, cạnh và rừng

- **Nút (node):** một ô chứa `val`, `left`, `right`.
- **Cạnh (edge):** liên kết từ cha tới con.
- **Cây con (subtree):** nút bất kỳ và các hậu duệ bên dưới nó.
- **Rừng (forest):** nhiều cây tách rời nhau, hoặc rừng rỗng nếu mọi nút bị xóa.

Quan trọng: **xóa 3 không có nghĩa xóa 6 và 7**. Nếu 6 và 7 chưa bị xóa, chúng trở thành gốc mới.

In [1]:
from collections import deque
from typing import List, Optional

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f'TreeNode({self.val})'


def build_tree(values):
    """Dựng cây theo thứ tự tầng của LeetCode; None nghĩa là thiếu nút."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    pending = deque([root])
    i = 1
    while pending and i < len(values):
        node = pending.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            pending.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            pending.append(node.right)
        i += 1
    return root


def serialize_tree(root):
    """Đổi cây thành list theo tầng, loại bỏ None dư ở cuối."""
    if root is None:
        return []
    result, pending = [], deque([root])
    while pending:
        node = pending.popleft()
        if node is None:
            result.append(None)
            continue
        result.append(node.val)
        pending.extend([node.left, node.right])
    while result and result[-1] is None:
        result.pop()
    return result


def forest_as_sorted_lists(roots):
    """So sánh rừng mà không phụ thuộc thứ tự các cây được trả về."""
    return sorted((serialize_tree(root) for root in roots), key=str)

### 3.1. Xây và đọc cây theo cách LeetCode

LeetCode biểu diễn cây theo **thứ tự tầng**; `None` (hay `null` trên LeetCode) chỉ vị trí không có nút. `build_tree` và `serialize_tree` ở đây chỉ phục vụ học và test, **không dán vào bài nộp**.

In [2]:
tree=build_tree([1,2,3,4,5,6,7])
print('Gốc:',tree,'trái:',tree.left,'phải:',tree.right)
print('Theo tầng:',serialize_tree(tree))
assert serialize_tree(build_tree([1,None,2,3]))==[1,None,2,3]

Gốc: TreeNode(1) trái: TreeNode(2) phải: TreeNode(3)
Theo tầng: [1, 2, 3, 4, 5, 6, 7]


## 4. Python căn bản: `None`, `is None` và thay đổi liên kết

`None` biểu thị không có nút. `node.left=None` **cắt liên kết** từ node tới con trái. Đây khác với gán `node.left.val=0` (vẫn giữ nút nhưng đổi giá trị). `if node` kiểm tra có nút hay không; `node.val` chỉ an toàn sau khi biết `node is not None`.

In [3]:
a=TreeNode(10, TreeNode(20), TreeNode(30))
print('Trước:', serialize_tree(a))
removed=a.left
a.left=None
print('Sau:', serialize_tree(a),'; nút tách ra:',removed)
assert serialize_tree(a)==[10,None,30]

Trước: [10, 20, 30]
Sau: [10, None, 30] ; nút tách ra: TreeNode(20)


## 5. Python nền tảng: `list` khác `set` khi kiểm tra thuộc tập

`x in to_delete` với **list** có thể phải kiểm tra từng phần tử (O(k) cho k phần tử). Dùng `delete_set=set(to_delete)` và `x in delete_set` có thời gian trung bình O(1), khi giá trị là số nguyên.

`set` chỉ chứa phần tử không trùng; trong bài này danh sách xóa đã khác nhau nhưng vẫn chuyển thành `set` để tra cứu nhanh.

In [4]:
to_delete=[3,5]
delete_set=set(to_delete)
for x in [1,3,5,7]:
    print(x,'cần xóa?',x in delete_set)
assert 3 in delete_set and 2 not in delete_set

1 cần xóa? False
3 cần xóa? True
5 cần xóa? True
7 cần xóa? False


## 6. Cách trực giác (baseline): DFS + tìm trong list

Khi duyệt tới một nút: trước hết xử lý hai cây con; sau đó nếu nút này bị xóa thì **đẩy các con còn sống thành gốc mới**, rồi trả về `None` để cha cắt liên kết. Nếu không bị xóa, trả chính nút đó.

Đây là **postorder DFS (trái → phải → nút)**. Bản baseline cố tình dùng `in to_delete` để thấy chỗ có thể tối ưu từ O(nk) xuống O(n).

In [5]:
def del_nodes_baseline(root, to_delete):
    answer=[]
    def dfs(node):
        if node is None:
            return None
        node.left=dfs(node.left)
        node.right=dfs(node.right)
        if node.val in to_delete:  # Tìm tuyến tính trong list.
            if node.left: answer.append(node.left)
            if node.right: answer.append(node.right)
            return None
        return node
    remaining=dfs(root)
    if remaining: answer.append(remaining)
    return answer

got=del_nodes_baseline(build_tree([1,2,3,4,5,6,7]), [3,5])
print('Các cây sau khi xóa:',forest_as_sorted_lists(got))
assert forest_as_sorted_lists(got)==forest_as_sorted_lists(
    [build_tree([1,2,None,4]),build_tree([6]),build_tree([7])])

Các cây sau khi xóa: [[1, 2, None, 4], [6], [7]]


### 6.1. `return None` có tác dụng gì?

Xét cha `2` có con phải `5`. Khi `dfs(5)` trả `None`, lệnh `node.right=dfs(node.right)` ở nút `2` tự động cắt cạnh `2→5`. Khi xóa nút `3`, hai con `6` và `7` (không bị xóa) trở thành hai gốc độc lập.

Vấn đề của baseline là: nếu `to_delete` có k phần tử, ta lặp tìm kiếm O(k) cho mỗi trong n nút → **O(nk)**.

## 7. Bước đột phá: dùng `set`, chỉ xử lý mỗi nút một lần

Ta thay duy nhất `node.val in to_delete` bằng `node.val in delete_set`.

**Bất biến sau lời gọi `dfs(node)`**: giá trị trả về là gốc cây con còn nối được với cha (hoặc `None` nếu gốc cũ bị xóa); các gốc mới sinh ra bên dưới đã được ghi vào `answer`.

Phải xử lý **con trước cha** để khi xóa cha, ta biết con nào vẫn còn sống.

In [6]:
def del_nodes_postorder(root, to_delete):
    deleted=set(to_delete)
    answer=[]
    def dfs(node):
        if node is None:
            return None
        node.left=dfs(node.left)
        node.right=dfs(node.right)
        if node.val in deleted:
            if node.left: answer.append(node.left)
            if node.right: answer.append(node.right)
            return None
        return node
    root=dfs(root)
    if root: answer.append(root)
    return answer

for values, removed in [([1,2,3,4,5,6,7],[3,5]),([1,2,4,None,3],[3])]:
    print(values,removed,'->',forest_as_sorted_lists(del_nodes_postorder(build_tree(values),removed)))

[1, 2, 3, 4, 5, 6, 7] [3, 5] -> [[1, 2, None, 4], [6], [7]]
[1, 2, 4, None, 3] [3] -> [[1, 2, 4]]


## 8. Trace: quan sát liên kết trước và sau cập nhật

Với `[1,2,3,4,5,6,7]`, xóa `[3,5]`:

| Nút đang hoàn tất | Xóa? | Con trả về sau DFS | Kết quả |
|---|:---:|---|---|
| 4 | Không | `(None,None)` | Giữ 4 |
| 5 | Có | `(None,None)` | Trả `None`; 2 mất con 5 |
| 2 | Không | `(4,None)` | Giữ 2→4 |
| 6, 7 | Không | `(None,None)` | Giữ nguyên |
| 3 | Có | `(6,7)` | Thêm gốc 6 và 7; trả `None` |
| 1 | Không | `(2,None)` | Thêm gốc 1 |

Nên phân biệt **thứ tự chạy DFS** và **thứ tự cây trong kết quả**: đề cho phép thứ tự bất kỳ.

In [7]:
def trace_delete(root, to_delete):
    deleted=set(to_delete)
    answer=[]
    def walk(node):
        if node is None: return None
        node.left=walk(node.left)
        node.right=walk(node.right)
        do_delete=node.val in deleted
        left=node.left.val if node.left else None
        right=node.right.val if node.right else None
        print(f'nút {node.val}: con sống={left,right}; xóa? {do_delete}')
        if do_delete:
            if node.left: answer.append(node.left)
            if node.right: answer.append(node.right)
            print('  gốc rừng lúc này:',[x.val for x in answer])
            return None
        return node
    head=walk(root)
    if head: answer.append(head)
    print('Các gốc cuối:',[x.val for x in answer])
    return answer

trace_delete(build_tree([1,2,3,4,5,6,7]),[3,5])

nút 4: con sống=(None, None); xóa? False
nút 5: con sống=(None, None); xóa? True
  gốc rừng lúc này: []
nút 2: con sống=(4, None); xóa? False
nút 6: con sống=(None, None); xóa? False
nút 7: con sống=(None, None); xóa? False
nút 3: con sống=(6, 7); xóa? True
  gốc rừng lúc này: [6, 7]
nút 1: con sống=(2, None); xóa? False
Các gốc cuối: [6, 7, 1]


[TreeNode(6), TreeNode(7), TreeNode(1)]

## 9. Cây lệch sâu: tại sao cần biết giải pháp không đệ quy?

**DFS đệ quy** sử dụng call stack của Python. Với cây có gần 1000 nút nối thành một chuỗi, môi trường Python có thể chạm giới hạn `RecursionError` (giới hạn phụ thuộc hệ thống). Đề cho phép tới 1000 nút, nên để chắc chắn ta có thể dùng **stack tự quản lý**.

Ý tưởng duyệt từ trên xuống: mỗi phần tử stack lưu `(node, parent_deleted)`.
- Nếu `parent_deleted=True` và **node còn sống** → node là một gốc mới.
- Nếu một cạnh nối tới nút bị xóa → cắt cạnh đó.
- Nếu node bị xóa, cắt các cạnh ra con của nó; nhưng **vẫn phải duyệt các con** để cứu những nút còn sống bên dưới.

In [8]:
def del_nodes_iterative(root, to_delete):
    if root is None: return []
    deleted=set(to_delete)
    forest=[]
    stack=[(root,True)]  # Gốc đầu tiên coi như có cha đã bị xóa.
    while stack:
        node,parent_deleted=stack.pop()
        is_deleted=node.val in deleted
        if parent_deleted and not is_deleted:
            forest.append(node)
        for side in ('left','right'):
            child=getattr(node,side)
            if child is None: continue
            stack.append((child,is_deleted))
            if is_deleted or child.val in deleted:
                setattr(node,side,None)
    return forest

assert forest_as_sorted_lists(del_nodes_iterative(build_tree([1,2,3,4,5,6,7]),[3,5])) == [
    [1,2,None,4],[6],[7]]
print('Iterative DFS ->',forest_as_sorted_lists(del_nodes_iterative(build_tree([1,2,3,4,5,6,7]),[3,5])))

Iterative DFS -> [[1, 2, None, 4], [6], [7]]


### 9.1. Giải thích cú pháp mới `getattr`, `setattr` và tuple

- `('left','right')`: một tuple chứa 2 tên thuộc tính.
- `getattr(node,'left')` tương đương `node.left`.
- `setattr(node,'right',None)` tương đương `node.right=None`.
- `stack.pop()` lấy phần tử cuối (LIFO): thao tác O(1) trung bình với list Python.

Đây chỉ là cách tránh viết hai khối code trái/phải giống nhau. Bạn có thể đổi lại thành hai khối `if node.left` và `if node.right` nếu dễ hiểu hơn.

In [9]:
node=TreeNode(9,TreeNode(8),TreeNode(7))
print('Đọc trái:',getattr(node,'left'))
setattr(node,'right',None)
print('Sau xóa cạnh phải:',serialize_tree(node))
stack=[('A',True),('B',False)]
print('pop ->',stack.pop(),'; còn:',stack)

Đọc trái: TreeNode(8)
Sau xóa cạnh phải: [9, 8]
pop -> ('B', False) ; còn: [('A', True)]


## 10. Code nộp LeetCode

Chọn **Python3** và nộp `class Solution` sau đây. Mã sử dụng **iterative DFS O(n)** để tránh mọi rủi ro từ cây quá sâu. Trên LeetCode, lớp `TreeNode` có sẵn theo đề nên không cần dán `build_tree`.

In [10]:
class Solution:
    def delNodes(self, root: Optional[TreeNode], to_delete: List[int]) -> List[TreeNode]:
        if root is None:
            return []
        deleted=set(to_delete)
        forest=[]
        stack=[(root,True)]
        while stack:
            node,parent_deleted=stack.pop()
            remove=node.val in deleted
            if parent_deleted and not remove:
                forest.append(node)
            for side in ('left','right'):
                child=getattr(node,side)
                if child is None:
                    continue
                stack.append((child,remove))
                if remove or child.val in deleted:
                    setattr(node,side,None)
        return forest

answer=Solution().delNodes(build_tree([1,2,3,4,5,6,7]),[3,5])
print('Ví dụ 1:',forest_as_sorted_lists(answer))
assert forest_as_sorted_lists(answer)==[[1,2,None,4],[6],[7]]

Ví dụ 1: [[1, 2, None, 4], [6], [7]]


## 11. Vì sao thuật toán đúng?

Đối với mỗi nút chưa xóa, có đúng hai trường hợp:
1. **Cha của nó cũng còn sống**: cạnh cha→con được giữ, nên nó không phải gốc mới.
2. **Nó là gốc ban đầu hoặc cha nó đã xóa**: `parent_deleted=True` → đưa nó vào `forest` **đúng một lần**.

Mọi cạnh có ít nhất một đầu mút bị xóa đều được cắt. Mọi cạnh nối hai nút sống đều giữ nguyên. Vì cây ban đầu không có chu trình, các thành phần liên thông còn lại đều là cây, và mỗi thành phần có đúng một gốc như đã ghi vào `forest`. Stack bảo đảm mỗi nút gốc từ cây ban đầu được thăm đúng một lần.

## 12. Độ phức tạp (định nghĩa rõ biến)

Gọi `n` là số nút cây, `k=len(to_delete)`.

| Phương án | Thời gian | Bộ nhớ bổ sung |
|---|---|---|
| DFS + kiểm tra list | O(nk) | O(h+n) kể cả kết quả, với h là chiều cao |
| DFS đệ quy + set | O(n+k) trung bình | O(n+k), tính cả kết quả và call stack |
| **DFS stack + set** | **O(n+k) trung bình** | **O(n+k)** kể cả tập xóa, stack và kết quả |

Mỗi nút vào stack một lần và tối đa hai cạnh được xét. Nếu bỏ qua danh sách kết quả, bộ nhớ vẫn có thể O(n) do stack + `set`. Hash set có thời gian kiểm tra **trung bình** O(1), không cam kết worst-case trong mọi tình huống.

## 13. Bộ test: mẫu, biên và so sánh hai cách

Mỗi lần test phải **dựng cây mới**, vì thuật toán cố ý sửa `left/right` tại chỗ. So rừng đã chuẩn hóa thay vì so thứ tự list các gốc, vì thứ tự không bắt buộc.

In [11]:
examples=[
    ([1,2,3,4,5,6,7],[3,5],[[1,2,None,4],[6],[7]]),
    ([1,2,4,None,3],[3],[[1,2,4]]),
    ([1],[],[[1]]),
    ([1],[1],[]),
    ([1,2,3],[1],[[2],[3]]),
    ([1,2,3],[1,2,3],[]),
    ([1,2,3,4,None,None,7],[2],[[1,None,3,None,7],[4]]),
]
for vals,rem,expected in examples:
    got=forest_as_sorted_lists(Solution().delNodes(build_tree(vals),rem))
    want=sorted(expected,key=str)
    assert got==want,(vals,rem,got,want)
print('PASS',len(examples),'test minh họa và biên')

PASS 7 test minh họa và biên


### 13.1. Oracle độc lập: xét từng cạnh của cây gốc

Để tránh 2 cách DFS cùng mắc một lỗi, ta xây **oracle riêng**: duyệt cây ban đầu, ghi cha của từng nút và những cạnh còn lại. Với mỗi nút còn sống, nếu cha không còn sống, đó là gốc. Sau đó tạo **cây sao chép** chỉ từ các nút sống. Oracle không dùng hàm nộp bài.

In [12]:
def oracle_forest(root, to_delete):
    if root is None: return []
    removed=set(to_delete)
    edges={}
    stack=[(root,None)]
    while stack:
        node,parent=stack.pop()
        edges[node.val]=(parent, node.left.val if node.left else None,
                         node.right.val if node.right else None)
        if node.left: stack.append((node.left,node.val))
        if node.right: stack.append((node.right,node.val))
    alive={v:TreeNode(v) for v in edges if v not in removed}
    roots=[]
    for value,(parent,left,right) in edges.items():
        if value not in alive: continue
        if parent is None or parent not in alive:
            roots.append(alive[value])
        alive[value].left=alive.get(left)
        alive[value].right=alive.get(right)
    return roots

a=build_tree([1,2,3,4,5,6,7])
assert forest_as_sorted_lists(oracle_forest(a,[3,5]))==[[1,2,None,4],[6],[7]]

In [13]:
import random
rng=random.Random(1110)
def random_tree(n):
    vals=list(range(1,n+1))
    rng.shuffle(vals)
    root=TreeNode(vals[0]); open_slots=[(root,'left'),(root,'right')]
    for val in vals[1:]:
        k=rng.randrange(len(open_slots))
        parent,side=open_slots.pop(k)
        node=TreeNode(val)
        setattr(parent,side,node)
        open_slots.extend([(node,'left'),(node,'right')])
    return root

for _ in range(200):
    size=rng.randint(1,25)
    base=random_tree(size)
    values=serialize_tree(base)
    remove=rng.sample(list(range(1,size+1)),rng.randint(0,size))
    expected=forest_as_sorted_lists(oracle_forest(build_tree(values),remove))
    got=forest_as_sorted_lists(Solution().delNodes(build_tree(values),remove))
    assert got==expected,(values,remove,got,expected)
print('PASS 200 cây sinh ngẫu nhiên, đối chiếu oracle độc lập')

PASS 200 cây sinh ngẫu nhiên, đối chiếu oracle độc lập


### 13.2. Cây lệch 1000 nút: test trường hợp đệ quy dễ gặp rủi ro

Tạo chuỗi `1→2→...→1000` chỉ qua con trái. Chọn nhiều nút để xóa; hàm không đệ quy phải chạy được mà không chỉnh `sys.setrecursionlimit`.

In [14]:
chain=TreeNode(1)
cur=chain
for value in range(2,1001):
    cur.left=TreeNode(value)
    cur=cur.left
removed=list(range(2,1001,2))
result=Solution().delNodes(chain,removed)
assert len(result)==500
assert {x.val for x in result}==set(range(1,1001,2))
assert all(x.left is None and x.right is None for x in result)
print('PASS cây lệch 1000 nút; còn',len(result),'cây')

PASS cây lệch 1000 nút; còn 500 cây


## 14. Sai lầm phổ biến

1. **Xóa luôn toàn bộ cây con:** sai, vì con của nút xóa có thể trở thành gốc mới.
2. **Chỉ xóa giá trị:** `node.val=0` vẫn giữ nút/cạnh; phải loại nút khỏi cây.
3. **Quên kiểm tra gốc ban đầu:** nếu gốc không xóa, nó cũng nằm trong rừng kết quả.
4. **So thứ tự cây trong rừng:** đề cho phép thứ tự bất kỳ.
5. **Test hai thuật toán trên cùng cây đã sửa:** lần thứ hai không còn cây gốc để đối chiếu.
6. **Dùng DFS đệ quy trên chuỗi rất sâu:** có thể chạm giới hạn đệ quy Python.

In [15]:
# Phản ví dụ của lỗi "xóa cả cây con khi xóa cha":
counterexample=build_tree([1,2,3])
result=Solution().delNodes(counterexample,[1])
print('Xóa 1 phải giữ hai cây:',forest_as_sorted_lists(result))
assert forest_as_sorted_lists(result)==[[2],[3]]

Xóa 1 phải giữ hai cây: [[2], [3]]


## 15. Nhận diện pattern & biến thể

- **Xóa / lọc nút** mà giữ lại các phần còn nối được → cắt cạnh và xét gốc thành phần liên thông.
- **Kết quả của con quyết định liên kết với cha** → nghĩ đến DFS postorder.
- **Cần trạng thái cha** khi duyệt từ trên xuống → truyền `parent_deleted`.
- **Nhiều giá trị cần kiểm tra thuộc tập** → dùng `set`.

Biến thể: trả về số cây còn lại; đếm nút của mỗi cây; chỉ lấy gốc có giá trị nhỏ nhất; chuyển bài sang cây N-nhánh.

In [16]:
def forest_sizes(roots):
    result=[]
    for root in roots:
        count=0;stack=[root]
        while stack:
            node=stack.pop();count+=1
            if node.left:stack.append(node.left)
            if node.right:stack.append(node.right)
        result.append(count)
    return sorted(result)

assert forest_sizes(Solution().delNodes(build_tree([1,2,3,4,5,6,7]),[3,5]))==[1,1,3]
print('Kích thước các cây:',[1,1,3])

Kích thước các cây: [1, 1, 3]


## 16. Bài tập (tự làm trước khi mở đáp án)

**Bài 1 — Tính tay (dễ):** Xóa `[2]` khỏi cây `[1,2,3,4,5]`; ghi các gốc rừng còn lại.

**Bài 2 — Dự đoán output:** `set([2,2,5])`, `2 in ...` và `4 in ...` in ra gì?

**Bài 3 — Điền TODO:** viết `exercise_count_survivors` đếm số nút không bị xóa trong cây.

**Bài 4 — Tự viết:** viết `exercise_del_nodes` trả về rừng bằng DFS, không nhìn `Solution`.

**Bài 5 — Biến thể:** viết `exercise_smallest_root` trả về giá trị gốc nhỏ nhất sau xóa, hoặc `None` nếu rừng rỗng.

In [17]:
def exercise_count_survivors(root, to_delete):
    # TODO 1: dùng set để kiểm tra nhanh.
    # TODO 2: duyệt mọi nút bằng stack.
    # TODO 3: tăng đếm khi giá trị chưa bị xóa.
    pass

def exercise_del_nodes(root, to_delete):
    # TODO: tự viết cách xử lý cạnh và gốc; có thể dùng postorder hoặc stack.
    pass

def exercise_smallest_root(root, to_delete):
    # TODO: tận dụng hàm trả rừng của bạn rồi lấy min(root.val).
    pass

# Tự làm xong mới bỏ dấu # để kiểm tra:
# assert exercise_count_survivors(build_tree([1,2,3]),[2])==2
# assert forest_as_sorted_lists(exercise_del_nodes(build_tree([1,2,3]),[1]))==[[2],[3]]
# assert exercise_smallest_root(build_tree([1,2,3]),[1])==2

## 17. Gợi ý theo 3 mức

| Bài | Hướng tư duy | Cú pháp | Gần-code |
|---|---|---|---|
| 3 | Đếm nút sống | `stack.pop()` | `if node.val not in deleted: count+=1` |
| 4 | Xóa nút, giữ cây con sống | DFS trả `node/None` | `node.left=dfs(node.left)`; nếu xóa thì thêm con sống |
| 5 | Lấy gốc nhỏ nhất | `min(..., default=None)` | `min((x.val for x in roots), default=None)` |

Bài 1: các nút `4,5` không mất chỉ vì cha `2` bị xóa. Bài 2: `set` loại giá trị lặp.

## 18. Đáp án tham khảo — chỉ xem sau khi tự làm

**Bài 1:** rừng `[[1,None,3],[4],[5]]` (thứ tự tùy ý).  
**Bài 2:** `{2,5}`, `True`, `False`.  
**Bài 3–5:** code bên dưới; không cần sửa cell TODO để phần minh họa trước chạy được.

In [18]:
def answer_count_survivors(root, to_delete):
    deleted=set(to_delete); count=0; stack=[root] if root else []
    while stack:
        node=stack.pop()
        count+=node.val not in deleted
        if node.left: stack.append(node.left)
        if node.right: stack.append(node.right)
    return count

def answer_del_nodes(root, to_delete):
    return del_nodes_iterative(root,to_delete)

def answer_smallest_root(root, to_delete):
    roots=answer_del_nodes(root,to_delete)
    return min((node.val for node in roots),default=None)

assert answer_count_survivors(build_tree([1,2,3]),[2])==2
assert forest_as_sorted_lists(answer_del_nodes(build_tree([1,2,3]),[1]))==[[2],[3]]
assert answer_smallest_root(build_tree([1,2,3]),[1])==2
print('Đáp án mẫu: PASS')

Đáp án mẫu: PASS


## 19. Ghi nhớ / checklist nộp bài

- [ ] Tôi phân biệt **xóa nút** và **xóa toàn bộ cây con**.
- [ ] Tôi kiểm tra `to_delete` bằng `set`, không tìm tuyến tính nhiều lần.
- [ ] Tôi biết khi nào một nút sống trở thành **gốc mới**.
- [ ] Tôi xử lý đúng cả trường hợp xóa gốc, xóa tất cả và không xóa gì.
- [ ] Tôi chỉ nộp `class Solution` với `delNodes` đúng chữ ký, không nộp hàm hỗ trợ test.
- [ ] Tôi biết code sửa cây tại chỗ nên phải dựng cây mới cho mỗi lần test.

**Mẫu tư duy:** “Cha bị xóa + con sống → con trở thành gốc mới; cạnh chạm nút bị xóa → cắt.”